Lab9: OSM and Interactive Web Maps
---
**Xiaofan Liang**  
**Created in 2024; Updated in 2026**


## Learning Objectives
By the end of this practical lab you will be able to:

* Retrieve and process OpenStreetMap (OSM) data with `OSMnx`
* Work with OSM features as `GeoPandas` GeoDataFrames
* Visualize OSM data as a multi-layer interactive map with `Folium`
* Export a Folium map as a static HTML file you can open in any browser
* Host the map on GitHub Pages so anyone can view it with a link

**CHEATSHEETS!** Useful references include the [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html), [GeoPandas documentation](https://geopandas.org/en/stable/docs/user_guide.html), [OSMnx documentation](https://osmnx.readthedocs.io/), [Folium User Guide](https://python-visualization.github.io/folium/latest/user_guide.html), [OSM Wiki Map Features](https://wiki.openstreetmap.org/wiki/Map_features), [Taginfo](https://taginfo.openstreetmap.org/), and [Leaflet Providers Preview](https://leaflet-extras.github.io/leaflet-providers/preview/).

| Function | Task |
|---|---|
| `ox.geocode_to_gdf()` | Retrieve a place boundary by place name and return it as a GeoDataFrame |
| `.total_bounds` | Retrieve a GeoDataFrame bounding box as `(xmin, ymin, xmax, ymax)` |
| `ox.features_from_bbox()` | Retrieve OSM features within a bounding box based on key-value tags |
| `geom_type` | Inspect whether features are points, lines, or polygons |
| `make_valid()` | Repair invalid geometries before mapping or spatial analysis |
| `GeoDataFrame.explore()` | Build an interactive Folium map directly from a GeoDataFrame |
| `explore(m=...)` | Add another GeoDataFrame as a layer onto an existing map |
| `explore(column=..., categorical=True)` | Color features by a category, with an automatic legend |
| `tooltip=` / `popup=` | Show feature info on hover / on click |
| `folium.LayerControl()` | Add controls for turning map layers on and off |
| `fit_bounds()` | Frame the map around a data extent |
| `m.save()` | Export the interactive map as an HTML file |

`OSMnx` queries OpenStreetMap features directly from the Overpass API using a place name or bounding box, and returns them as GeoDataFrames. `Folium` builds interactive Leaflet web maps from those GeoDataFrames and exports them as standalone HTML, which makes them easy to host on a static site like GitHub Pages.


## Environment, Libraries, and Data

Lab 1 created the course virtual environment, and Lab 2 reviewed how to activate it and select its notebook kernel. **Activate the same environment before opening this notebook.**

Install the Lab 9 packages (once) in that environment:
```bash
python3 -m pip install osmnx folium
```

What new packages are for:

* **osmnx** -- queries OpenStreetMap features (buildings, roads, amenities, ...) directly from the Overpass API and returns them as GeoDataFrames.
* **folium** -- builds interactive Leaflet web maps in Python and exports them as standalone HTML.

This lab queries OpenStreetMap data live for the University of Michigan North Campus area -- no local data files are needed.


In [3]:
%pip install osmnx

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
from shapely import make_valid

# Cache OSMnx requests so repeated runs don't keep re-downloading the same data.
ox.settings.use_cache = True

## Retrieve OSM Data on UMich North Campus

OSM data has three geometry types: **Node**, **Way**, and **Relation**. A node can be a point of interest, but it can also be a vertex that forms the geometry of a polygon. A way can be a line or polygon, such as a road or building. A relation is a collection of nodes, ways, and other relations, such as a bus line or administrative boundary. In this lab, we mostly work with nodes and ways.

There are two steps for downloading OSM data:

1. Determine the **bounding box** for the download.
2. Tell OSM what **features** (key-value tags) you want -- amenities, roads, buildings -- and convert the result into a GeoDataFrame.


In [5]:


# Configure OSMnx to use another Overpass endpoint
ox.settings.overpass_url = "https://overpass-api.de/api"

# Increase timeout
ox.settings.timeout = 300

# Keep caching enabled
ox.settings.use_cache = True

print("Overpass settings updated!")



Overpass settings updated!


### Step 1: Determine the Bounding Box

An OSM query starts with a bounding box. You can pass a place name as a string: `ox.geocode_to_gdf()` uses a geocoding service to find the boundary associated with a place name and returns it as a GeoDataFrame, and `.total_bounds` then gives you `(xmin, ymin, xmax, ymax)`. Always double-check the returned boundary since many places share similar names.

You can also supply four coordinates directly as `(xmin, ymin, xmax, ymax)`. Click the bottom-left corner of your area of interest in Google Maps (or another map interface) for `xmin`/`ymin`, and the top-right corner for `xmax`/`ymax`. Since "North Campus" isn't a well-defined place name in OSM, we use a numeric bounding box for it instead.


In [6]:
# Geocode Ann Arbor and the University of Michigan for comparison -- neither perfectly
# matches North Campus, which is why we use a numeric bounding box below instead.
aa = ox.geocode_to_gdf('Ann Arbor, Michigan, USA')
print('Ann Arbor total_bounds:', aa.total_bounds)

um = ox.geocode_to_gdf('University of Michigan, Ann Arbor, Michigan, USA')
print('UMich total_bounds:', um.total_bounds)

# Roughly the bottom-left and top-right corners of UM North Campus.
# OSMnx expects bbox values in the order (left, bottom, right, top).
bbox = (-83.725706, 42.287305, -83.704518, 42.302508)
bbox

Ann Arbor total_bounds: [-83.799572   42.222668  -83.675807   42.3238941]
UMich total_bounds: [-83.7502236  42.242226  -83.6649743  42.3125148]


(-83.725706, 42.287305, -83.704518, 42.302508)

### Step 2: Add OSM Features and Convert to GeoDataFrames

OSM features are stored as **key-value** (or **feature-value**) pairs. To retrieve restaurants, you'd use `key='amenity'`, `value='restaurant'` -- written in OSMnx as a dictionary, `{"amenity": "restaurant"}`. Passing `True` instead of a value (e.g. `{"building": True}`) retrieves **all** values under that key; passing a list (e.g. `{"amenity": ["restaurant", "cafe"]}`) retrieves several values in one query.

**How do you find out what key-value pairs exist?**

- The [OSM Wiki Map Features page](https://wiki.openstreetmap.org/wiki/Map_features) covers most commonly used pairs, with definitions for each.
- [Taginfo](https://taginfo.openstreetmap.org/) shows how many instances of a key-value pair exist -- useful for checking whether a tag is common enough, or is only used in a specific local context.
- Google your interest together with "OSM." Searching "restaurant in OSM" turns up the [amenity=restaurant](https://wiki.openstreetmap.org/wiki/Tag:amenity%3Drestaurant) page directly.

We retrieve three layers this way: **buildings**, **roads**, and **amenities**.


#### Buildings

`{"building": True}` retrieves every building-tagged feature. The result can mix point and polygon geometries (a node may carry building-related tags even though the polygon is what we want), so we keep only polygons. OSM polygons occasionally have minor topology errors (such as not closing correctly); `make_valid()` from `shapely` repairs those before mapping. We also drop down to just the `geometry` column. OSM features come back with dozens of tag columns we don't need for this map, and keeping them increases the size of the exported file.


In [7]:
# Retrieve all features within the bounding box that are tagged as buildings.
building = ox.features_from_bbox(bbox, tags={'building': True})

# Print the count of each geometry type 
print(building.geom_type.value_counts())

# Fix any invalid geometries in the building GeoDataFrame and display the first few rows. make_valid() requires shapely 2.0 or later.
building['geometry'] = make_valid(building['geometry'])
display(building.head())

# reset row index to numbers and drop other columns; we only want to display limited information in the interactive map. 
building = building[['geometry']].reset_index(drop=True)
building.head()

Polygon         366
MultiPolygon      2
Name: count, dtype: int64


geometry  \
element  id                                                           
relation 3874939  POLYGON ((-83.72129 42.29726, -83.72117 42.297...   
         3874940  POLYGON ((-83.72088 42.29643, -83.72085 42.295...   
         3874941  POLYGON ((-83.72084 42.29492, -83.72111 42.294...   
         3877244  POLYGON ((-83.71756 42.29006, -83.71676 42.290...   
         3883610  POLYGON ((-83.71168 42.29468, -83.71168 42.294...   

                  addr:city addr:housenumber addr:postcode addr:state  \
element  id                                                             
relation 3874939  Ann Arbor             1760         48105         MI   
         3874940  Ann Arbor             1770         48105         MI   
         3874941  Ann Arbor             1780         48105         MI   
         3877244  Ann Arbor             2000         48109         MI   
         3883610  Ann Arbor             2455           NaN         MI   

                          addr:street amenity    building building:levels  \
element  id                                                                 
relation 3874939      Broadway Street     NaN  apartments               4   
         3874940      Broadway Street     NaN  apartments               4   
         3874941      Broadway Street     NaN  apartments               4   
         3877244  Bonisteel Boulevard     NaN  university               3   
         3883610       Hayward Street     NaN  university               3   

                 community_centre  ... drinking_water internet_access  \
element  id                        ...                                  
relation 3874939              NaN  ...            NaN             NaN   
         3874940              NaN  ...            NaN             NaN   
         3874941              NaN  ...            NaN             NaN   
         3877244              NaN  ...            NaN             NaN   
         3883610              NaN  ...            NaN             NaN   

                 internet_access:fee internet_access:ssid leisure membership  \
element  id                                                                    
relation 3874939                 NaN                  NaN     NaN        NaN   
         3874940                 NaN                  NaN     NaN        NaN   
         3874941                 NaN                  NaN     NaN        NaN   
         3877244                 NaN                  NaN     NaN        NaN   
         3883610                 NaN                  NaN     NaN        NaN   

                 sauna sport swimming_pool toilets:wheelchair  
element  id                                                    
relation 3874939   NaN   NaN           NaN                NaN  
         3874940   NaN   NaN           NaN                NaN  
         3874941   NaN   NaN           NaN                NaN  
         3877244   NaN   NaN           NaN                NaN  
         3883610   NaN   NaN           NaN                NaN  

[5 rows x 106 columns]

,geometry
0,"POLYGON ((-83.72129 42.29726, -83.72117 42.297..."
1,"POLYGON ((-83.72088 42.29643, -83.72085 42.295..."
2,"POLYGON ((-83.72084 42.29492, -83.72111 42.294..."
3,"POLYGON ((-83.71756 42.29006, -83.71676 42.290..."
4,"POLYGON ((-83.71168 42.29468, -83.71168 42.294..."


#### Roads

Roads are stored under the `highway` key. In OSM, `highway` covers all road-like features -- residential streets, service roads, footways, cycleways, paths -- not just highways in the everyday sense. `{"highway": True}` retrieves all of them; we keep the line geometries and, again, trim to just the columns the map actually uses.


In [8]:
# retrieve all features within the bounding box that are tagged as highways. 
roads = ox.features_from_bbox(bbox, tags={'highway': True})

# print the count of each geometry type 
print(roads.geom_type.value_counts())

# only keep line geometries
roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])]

# Inspect the most common highway values in this area before trimming columns.
print(roads['highway'].value_counts().head(8))

# reset row index and keep the highway and geometry columns while dropping other columns; we only want to display limited information in the interactive map. 
roads = roads[['highway', 'geometry']].reset_index(drop=True)
roads.head()

LineString    1946
Point          342
Polygon         44
Name: count, dtype: int64
highway
footway         1271
service          358
steps             93
cycleway          63
unclassified      35
tertiary          33
path              32
residential       30
Name: count, dtype: int64


,highway,geometry
0,unclassified,"LINESTRING (-83.7127 42.30039, -83.71282 42.30..."
1,tertiary,"LINESTRING (-83.71297 42.29036, -83.71313 42.2..."
2,service,"LINESTRING (-83.72461 42.29701, -83.72525 42.2..."
3,service,"LINESTRING (-83.71025 42.29122, -83.71035 42.2..."
4,service,"LINESTRING (-83.7127 42.29956, -83.71278 42.29..."


#### Amenities

We retrieve several food-and-drink amenity types in one query by passing a list of values under `amenity`, then keep point geometries. Many OSM points are missing a `name`, which would leave the map's tooltip/popup blank. We drop those here to keep the demo map focused on labeled points, though in a real analysis you'd usually keep unnamed points too. 


In [9]:

# Define OSM tags to search for specific amenity types (food/drink, etc.)
tags = {
    'amenity': [
        'restaurant', 'bar', 'biergarten', 'fast_food',
        'food_court', 'cafe', 'pub', 'ice_cream',
    ]
}

# Retrieve all features within the bounding box that match the tags.
try:
    amenity = ox.features_from_bbox(bbox, tags=tags)

    # Print the count of each geometry type
    print(amenity.geom_type.value_counts())

    # Convert non-point geometries (Polygon, MultiPolygon) to centroids before filtering columns.
    amenity_points = amenity[amenity.geom_type == "Point"].copy()
    amenity_polys = amenity[
        amenity.geom_type.isin(["Polygon", "MultiPolygon"])
    ].copy()

    amenity_polys["geometry"] = amenity_polys.geometry.centroid

    amenity_points = pd.concat(
        [amenity_points, amenity_polys],
        ignore_index=True
    )

    # Drop entries missing the 'name' column
    amenity_points = amenity_points.dropna(subset=['name'])

    # Keep only relevant columns
    amenity_point = amenity_points[
        ['name', 'amenity', 'geometry']
    ].reset_index(drop=True)

    # Display a count for each amenity type
    print(amenity_point['amenity'].value_counts())

    print("Amenities successfully retrieved!")

except Exception as e:
    print("Amenities could not be retrieved.")
    print("Server error:", e)


Amenities could not be retrieved.
Server error: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and the request queue is full.

## What to do

Retry your request using exponential backoff: wait a short interval, then retry.
If the request fails again, double the wait time and retry again.

## For heavy or automated use

This public endpoint is intended for browser-based and human-operated tools.
For automated or high-volume use, run a local Overpass API instance:

https://github.com/b1tw153/Overpass-API/blob/main/README.md



In [10]:

# CartoDB Positron: a light, low-contrast basemap that keeps colored overlays readable.
# The {z}/{x}/{y}/{r} placeholders are filled in by Leaflet itself at render time,
# not by Python, so they stay as plain text here -- only the API key is substituted.
tile_url = f'https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png?key={CARTO_API_KEY}'

# required attribution for using the map tiles
attr = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

# Build the basemap as its own TileLayer
base_tiles = folium.TileLayer(tiles=tile_url, attr=attr, name='CartoDB Positron', control=False)

# Buildings first: this call creates the map itself (m), styled as solid dark grey polygons with no border.
m = building.explore(
    tiles=base_tiles,
    color='darkgrey',
    style_kwds={'fillOpacity': 0.8, 'weight': 0},
    name='buildings',
)

# Roads: pass m= to add this layer onto the SAME map instead of creating a new one.
roads.explore(
    m=m,
    color='black',
    style_kwds={'weight': 1.5, 'opacity': 0.7},
    name='roads',
)

# Amenities: color automatically by the 'amenity' column (categorical=True gives each
# unique value its own color from the cmap), with an automatic legend, and a
# tooltip (hover) / popup (click) showing the name and type for each point.
amenity_point.explore(
    m=m,
    column='amenity',
    categorical=True,
    cmap=['red', 'blue', 'darkgreen'],
    marker_kwds={'radius': 5},
    tooltip=['name', 'amenity'],
    popup=['name', 'amenity'],
    legend=True,
    name='amenities',
)

# Let users toggle the buildings, roads, and amenities layers independently.
folium.LayerControl(collapsed=False).add_to(m)

m



NameError: name 'CARTO_API_KEY' is not defined

## Visualize OSM Data

**Folium** is aimed at *interactive* web maps: popups, pan/zoom, a shareable HTML file. **GeoPandas + Matplotlib** (used throughout earlier labs) is the usual choice for *static*, publication-quality figures. `GeoDataFrame.explore()` is a fast bridge between the two: it builds a Folium map directly from a GeoDataFrame, in one line, for a quick interactive look before you invest in custom styling.


In [ ]:
# Fastest possible interactive preview: one GeoDataFrame, one line.
amenity_point.explore(color='blue', tiles='OpenStreetMap')

### Get a Free CARTO API Key

CARTO's basemap tiles (used below for their light, low-contrast look) now require a free API key -- unauthenticated requests come back with a visible "API KEY REQUIRED" watermark across the map. Getting a key is quick and does **not** require creating a CARTO account:

1. Go to [carto.com/basemaps/apikey](https://carto.com/basemaps/apikey).
2. Fill out the short form: your email, the domain where you'll host the map (your GitHub Pages domain is fine, e.g. `your-username.github.io`), and a one-line project description (e.g. "class assignment").
3. CARTO emails you a key, usually within a few minutes. The free tier covers 5 million tile requests/month -- far more than this lab needs.
4. Paste your key into `CARTO_API_KEY` below.

**Why CARTO and not Mapbox?** Mapbox also issues free API tokens, and token creation there is instant (no email wait at all) -- but Mapbox requires a credit card on file to unlock its free tier, even if you're never charged. CARTO's signup needs no card. For a whole class getting a key, that's worth more than saving a few minutes.


In [ ]:
# Replace with your own key from carto.com/basemaps/apikey 
CARTO_API_KEY = 'cb1_4kmi_1_9600d50c7d375def0ac2e908'

### Build the full map

`GeoDataFrame.explore()` builds a Folium map directly from a GeoDataFrame. Pass `m=` to add another GeoDataFrame onto a map you already built, instead of hand-building `folium.Map()` plus `folium.GeoJson()`/`CircleMarker()` calls yourself. We chain three layers onto one map:

- **buildings** -- solid dark grey polygons (this call creates the map)
- **roads** -- black lines
- **amenities** -- circle markers colored automatically by the `amenity` column (`column='amenity', categorical=True`), with an automatic legend, and a tooltip/popup showing the name and type

Each `.explore()` call's `name=` becomes one entry in the layer control. 

Folium can do much more than this! Explore its [User Guide](https://python-visualization.github.io/folium/latest/user_guide.html) for many more powerful examples, such as:
- [Choropleth maps](https://python-visualization.github.io/folium/latest/user_guide/geojson/choropleth.html): color polygons or regions by a data variable (e.g. population density, rates, etc.)
- [Heatmaps](https://python-visualization.github.io/folium/latest/user_guide/plugins/heatmap.html): visualize point intensity or density with smooth color gradients
- [TimestampedGeoJson](https://python-visualization.github.io/folium/latest/user_guide/plugins/timestamped_geojson.html): show how patterns change over time with a time slider
- Custom [popups](https://python-visualization.github.io/folium/latest/user_guide/ui_elements/popups.html), [icons](https://python-visualization.github.io/folium/latest/user_guide/ui_elements/icons.html), and [image overlays](https://python-visualization.github.io/folium/latest/user_guide/raster_layers/image_overlay.html)
- [GeoJSON with style functions and highlight features](https://python-visualization.github.io/folium/latest/user_guide/geojson/geojson.html)

You'll find many quick examples and patterns in the documentation!


In [ ]:

# CartoDB Positron: a light, low-contrast basemap that keeps colored overlays readable.
# The {z}/{x}/{y}/{r} placeholders are filled in by Leaflet itself at render time,
# not by Python, so they stay as plain text here -- only the API key is substituted.
tile_url = f'https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png?key={CARTO_API_KEY}'

# required attribution for using the map tiles
attr = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

# Build the basemap as its own TileLayer
base_tiles = folium.TileLayer(tiles=tile_url, attr=attr, name='CartoDB Positron', control=False)

# Buildings first: this call creates the map itself (m), styled as solid dark grey polygons with no border.
m = building.explore(
    tiles=base_tiles,
    color='darkgrey',
    style_kwds={'fillOpacity': 0.8, 'weight': 0},
    name='buildings',
)

# Roads: pass m= to add this layer onto the SAME map instead of creating a new one.
roads.explore(
    m=m,
    color='black',
    style_kwds={'weight': 1.5, 'opacity': 0.7},
    name='roads',
)

# Amenities: color automatically by the 'amenity' column (categorical=True gives each
# unique value its own color from the cmap), with an automatic legend, and a
# tooltip (hover) / popup (click) showing the name and type for each point.
amenity_point.explore(
    m=m,
    column='amenity',
    categorical=True,
    cmap=['red', 'blue', 'darkgreen'],
    marker_kwds={'radius': 5},
    tooltip=['name', 'amenity'],
    popup=['name', 'amenity'],
    legend=True,
    name='amenities',
)

# Let users toggle the buildings, roads, and amenities layers independently.
folium.LayerControl(collapsed=False).add_to(m)

m



NameError: name 'amenity_point' is not defined

Zoom, pan, and click markers to open popups. Use the layer control (top right) to toggle buildings, roads, and amenities.


## Export and Host on GitHub Pages

`m.save()` writes a standalone HTML file. If you double-click on the file exported in your local folder, you can see and interact with it on your browser. This is a local copy, meaning that you can see it, but you cannot share a link for others to see. To make it sharable, you will have to host it online and we recommend **GitHub Pages** because it provides free hosting service. 


In [ ]:
html_file = 'lab9_map.html'
m.save(html_file)
print('Saved:', html_file)

Saved: lab9_map.html


### Step 1: Create a GitHub repository

1. Log in to [GitHub](https://github.com) and click **New repository**.
2. Choose a name (e.g. `lab9-osm-map`), set visibility to **Public**, and create the repo.

### Step 2: Add your HTML file and enable GitHub Pages

**Option A -- upload in the browser**

1. In the repo, under the **Quick setup** ssection click the **upload an existing file** link and drag `lab9_map.html` into the upload box that opens.
2. Commit.
3. **Settings** -> **Pages** -> **Deploy from a branch** -> branch **main**, folder **/ (root)** -> Save.
4. After a short wait, open `https://<your-username>.github.io/<repo-name>/lab9_map.html`.

**Option B -- Git from your computer**

```bash
cd "PATH TO YOUR CURRENT LAB FOLDER"
git clone https://github.com/your-username/lab9-osm-map.git
cd lab9-osm-map
#  manually copy lab9_map.html into this folder, then type the following commands
git add lab9_map.html
git commit -m "Add Folium map"
git push origin main
```

Then enable Pages as in Option A.

### Step 3: Share the link

Use `https://<your-username>.github.io/<repo-name>/lab9_map.html` in assignments. **Tip:** rename the file to `index.html` in the repo root if you want the map at `https://<your-username>.github.io/<repo-name>/` with no filename.

## Practice Questions

### Q1: Retrieve a new OSM feature

Choose another OSM feature on North Campus -- bicycle parking, bus stops, parking lots, benches, or another amenity -- and retrieve it with `ox.features_from_bbox()`, following the same two-step process as above (the `bbox` from Step 1 still applies). Use the OSM Wiki or Taginfo to find the right key-value pair.





### Q2: Add it to the map

Add the new feature to the map above as another chained layer (besides amenities, roads, and building). Give it a color distinct from the amenities layer, and a tooltip or popup showing at least one useful attribute. Make sure it shows up as a new legend item and keep all other map aesthetics (e.g., legend style, basemap style, etc.) the same. 

### Q3: Export and host

Export your updated map as an HTML file, host it on GitHub Pages (or another static host), and submit the public URL for the quiz.

## Additional Resources

[OSMnx documentation](https://osmnx.readthedocs.io/), [Pyrosm documentation](https://pyrosm.readthedocs.io/) (an alternative for file-based OSM extracts), [GeoPandas documentation](https://geopandas.org/), [Folium User Guide](https://python-visualization.github.io/folium/latest/user_guide.html), [OSM Wiki Map Features](https://wiki.openstreetmap.org/wiki/Map_features), [Taginfo](https://taginfo.openstreetmap.org/), and [Leaflet Providers Preview](https://leaflet-extras.github.io/leaflet-providers/preview/).


In [ ]:


# ==========================================
# Q1: RETRIEVE BICYCLE PARKING
# ==========================================

# Retrieve bicycle parking from OpenStreetMap
bicycle_tags = {'amenity': 'bicycle_parking'}

bicycle_parking = ox.features_from_bbox(
    bbox,
    tags=bicycle_tags
)

# Convert polygon geometries to centroids
bicycle_parking = bicycle_parking.copy()

bicycle_parking['geometry'] = bicycle_parking.geometry.apply(
    lambda geom: geom.centroid
    if geom.geom_type in ['Polygon', 'MultiPolygon']
    else geom
)

# Keep point geometries
bicycle_point = bicycle_parking[
    bicycle_parking.geom_type == 'Point'
].copy()

bicycle_point = bicycle_point[
    ['amenity', 'geometry']
].reset_index(drop=True)

print("Bicycle parking features found:", len(bicycle_point))


# ==========================================
# Q2: BUILD THE ORIGINAL MAP AND ADD THE NEW LAYER
# ==========================================

# CARTO API KEY
CARTO_API_KEY = 'cb1_4kmi_1_9600d50c7d375def0ac2e908'

# CartoDB Positron basemap
tile_url = f'https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png?key={CARTO_API_KEY}'

attr = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

base_tiles = folium.TileLayer(
    tiles=tile_url,
    attr=attr,
    name='CartoDB Positron',
    control=False
)

# BUILDINGS
m = building.explore(
    tiles=base_tiles,
    color='darkgrey',
    style_kwds={'fillOpacity': 0.8, 'weight': 0},
    name='buildings'
)

# ROADS
roads.explore(
    m=m,
    color='black',
    style_kwds={'weight': 1.5, 'opacity': 0.7},
    name='roads'
)

# AMENITIES
amenity_point.explore(
    m=m,
    column='amenity',
    categorical=True,
    cmap=['red', 'blue', 'darkgreen'],
    marker_kwds={'radius': 5},
    tooltip=['name', 'amenity'],
    popup=['name', 'amenity'],
    legend=True,
    name='amenities'
)

# BICYCLE PARKING
if len(bicycle_point) > 0:
    bicycle_point.explore(
        m=m,
        color='purple',
        marker_kwds={'radius': 6},
        tooltip=['amenity'],
        popup=['amenity'],
        legend=False,
        name='bicycle parking'
    )
else:
    print("No bicycle parking features found in this area.")

# LAYER CONTROL
folium.LayerControl(collapsed=False).add_to(m)

# DISPLAY MAP
m


# ==========================================
# Q3: EXPORT UPDATED MAP
# ==========================================

html_file = 'lab9_map.html'

m.save(html_file)

print("Your updated map has been saved as:", html_file)


Bicycle parking features found: 136


NameError: name 'amenity_point' is not defined